## 모델의 기능 확장하기

LLM은 매우 강력한 도구이지만, 그 능력은 결국 학습된 지식이나 정보의 범위에 한정됩니다.  
결국, "알고 있는 것만 알 수 있다"는 말이죠.  
그렇다면, 학습 데이터에 없는 질문을 해야 할 때는 어떻게 해야 할까요?  
혹은 학습 데이터에는 없지만, 관련된 정보를 묻고 싶다면요?

이 문제를 해결하는 방법은 여러 가지가 있으며, 사용 가능한 리소스나 시간, 예산에 따라 선택할 수 있습니다.  
다음은 몇 가지 주요 옵션입니다:

- 필요한 정보를 포함하도록 모델을 **완전히 재학습**시키는 방법.  
  하지만 LLM을 재학습시키는 것은 전 세계에서도 소수의 기업만이 감당할 수 있는 작업으로, 수천 개의 GPU를 몇 주간 가동해야 가능한 수준입니다.

- 새로운 정보를 반영하여 모델을 **파인튜닝(fine-tuning)** 하는 방법.  
  이는 훨씬 적은 리소스로도 가능하며, 보통 몇 분 또는 몇 시간 안에 완료할 수 있습니다 (모델 크기에 따라 다름).  
  단점은 전체 모델을 다시 학습시키는 것이 아니기 때문에 새로운 정보가 답변에 완전히 반영되지 않을 수 있다는 점입니다.  
  파인튜닝은 특정 문맥이나 용어에 대한 이해도를 높이는 데는 뛰어나지만, 새로운 지식을 삽입하는 데는 한계가 있습니다.  
  또한 정보를 추가할 때마다 모델을 다시 학습하고 배포해야 합니다.

- 새로운 정보를 **데이터베이스**에 저장하고, 질의에 관련된 정보를 해당 질의의 **문맥(Context)** 으로 함께 첨부하여 LLM에 전달하는 방법.  
  이 기술을 **RAG(Retrieval Augmented Generation)** 라고 합니다.  
  RAG의 장점은, 모델을 재학습하거나 파인튜닝하지 않아도 새로운 지식을 활용할 수 있으며, 언제든지 쉽게 갱신할 수 있다는 점입니다.

우리는 이미 [Milvus](https://milvus.io/)를 사용해 **벡터 데이터베이스(Vector Database)** 를 준비해 두었습니다.  
이 데이터베이스에는 [California Driver's Handbook](https://www.dmv.ca.gov/portal/handbook/california-driver-handbook/)의 내용을  
[임베딩(Embeddings)](https://www.ibm.com/topics/embedding) 형태로 저장해 두었습니다.

이번 노트북에서는 RAG 기법을 활용하여 **보험 청구에 대한 질문을 수행**하고,  
LLM 자체를 수정하지 않고도 새로운 지식이 어떻게 도움이 될 수 있는지를 실습해보겠습니다.

### 요구사항 및 라이브러리 임포트

실습 지침에 따라 올바른 워크벤치 이미지를 선택하여 실행하였다면, 필요한 모든 라이브러리가 이미 설치되어 있을 것입니다.  
그렇지 않은 경우에는 다음 셀의 첫 번째 줄 주석을 해제하여 필요한 패키지를 설치하세요.

In [ ]:
# 아래 줄은 올바른 워크벤치 이미지를 선택하지 않았거나, 이 노트북을 워크숍 환경 외부에서 사용하는 경우에만 주석을 해제하십시오.
# !pip install --no-cache-dir --no-dependencies --disable-pip-version-check -r requirements.txt

import json

import transformers
from langchain.callbacks.streaming_stdout import StreamingStdOutCallbackHandler
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from langchain_community.llms import VLLMOpenAI
from langchain_huggingface import HuggingFaceEmbeddings
from milvus_retriever_with_score_threshold import \
    MilvusRetrieverWithScoreThreshold

# Turn off warnings when downloading the embedding model
transformers.logging.set_verbosity_error()

### Langchain 구성 요소

이번에도 Langchain을 사용하여 작업 파이프라인을 정의할 것입니다.

먼저, 질의를 보낼 **LLM**을 정의합니다.

In [ ]:
# LLM Inference Server URL
inference_server_url = "http://granite-7b-instruct-predictor.ic-shared-llm.svc.cluster.local:8080"

# LLM definition
llm = VLLMOpenAI(           # 우리는 vLLM OpenAI 호환 API 클라이언트를 사용하고 있습니다. 하지만 모델은 OpenAI가 아니라 OpenShift AI에서 실행되고 있습니다.
    openai_api_key="EMPTY",   # 따라서 OpenAI 키가 필요하지 않습니다.
    openai_api_base= f"{inference_server_url}/v1",
    model_name="granite-7b-instruct",
    top_p=0.92,
    temperature=0.01,
    max_tokens=512,
    presence_penalty=1.03,
    streaming=True,
    callbacks=[StreamingStdOutCallbackHandler()]
)

그 다음은 **벡터 데이터베이스**와의 연결입니다. 이 데이터베이스에는 California Driver Handbook의 내용이 사전에 준비되어 저장되어 있습니다.

In [ ]:
# 먼저, Handbook을 처리할 때 사용한 임베딩(embedding)을 정의합니다.  
model_kwargs = {"trust_remote_code": True}
embeddings = HuggingFaceEmbeddings(
            model_name="nomic-ai/nomic-embed-text-v1",
            model_kwargs=model_kwargs,
            show_progress=False,
        )

# 그런 다음, Milvus 벡터 저장소에서 관련 데이터를 검색할 retriever를 정의합니다.
retriever = MilvusRetrieverWithScoreThreshold(
            embedding_function=embeddings,
            collection_name="california_driver_handbook_1_0",
            collection_description="",
            collection_properties=None,
            connection_args={
                "host": "vectordb-milvus.ic-shared-milvus.svc.cluster.local",
                "port": "19530",
                "user": "root",
                "password": "Milvus",
            },
            consistency_level="Session",
            search_params=None,
            k=4,
            score_threshold=0.99,
            enable_dynamic_field=True,
            text_field="page_content",
        )

이제 질의를 수행할 때 사용할 **템플릿**을 정의하겠습니다.  

* 당신은 "Parasol Assistant"라는 이름의 도움이 되고, 정중하며, 정직한 어시스턴트입니다.  
* 당신은 보험 청구 요약, 참조 정보, 그리고 질문을 제공받게 됩니다.  
* **당신은 참조 정보를 활용하여, 최대한 해당 청구를 기반으로 질문에 답변해야 합니다.**
* 항상 가능한 한 유용하고, 안전하게 응답해야 합니다.  
* 당신의 답변에는 해롭거나, 비윤리적이거나, 인종차별적이거나, 성차별적이거나, 독성 있거나, 위험하거나, 불법적인 내용이 포함되어서는 안 됩니다.  
* 답변은 반드시 사회적으로 편향되지 않고, 긍정적인 성격을 유지해야 합니다.

* 질문이 말이 되지 않거나 사실적으로 일관되지 않는 경우에는, 틀린 답변을 하지 말고 그 이유를 설명해야 합니다.  
* 질문에 대한 답을 모를 경우에는, 잘못된 정보를 제공하지 말아야 합니다.

이번 템플릿에는 **References**(참조 문서) 섹션이 포함되어 있다는 점에 유의하세요.
벡터 데이터베이스로부터 반환된 문서들은 이 섹션에 삽입됩니다.

In [ ]:
template="""<|system|>
You are a helpful, respectful and honest assistant named "Parasol Assistant".
You will be given a claim summary, references to provide you with information, and a question.
You must answer the question based as much as possible on this claim with the help of the references.
Always answer as helpfully as possible, while being safe.
Your answers should not include any harmful, unethical, racist, sexist, toxic, dangerous, or illegal content.
Please ensure that your responses are socially unbiased and positive in nature.

If a question does not make any sense, or is not factually coherent, explain why instead of answering something not correct.
If you don't know the answer to a question, please don't share false information.

<|user|>
Claim Summary:
{claim}

References:
{{context}}

Question: {{question}}
<|assistant|>
"""


이제 모델에 질의할 준비가 완료되었습니다!

`claims` 폴더에는 실제로 수신될 수 있는 보험 청구 예시가 담긴 JSON 파일들이 있습니다.  
이제 첫 번째 청구를 읽고, 이에 관련된 질문을 해보겠습니다.

In [ ]:
# Read the claim and put its content in the "claim" variable

filename = 'claims/claim1.json'

# Opening JSON file
with open(filename, 'r') as file:
    data = json.load(file)
claim = data["content"]

### 첫 번째 테스트: 추가 지식 없이

먼저 벡터 데이터베이스의 도움 없이 보험 청구에 대한 첫 번째 질의를 수행해 보겠습니다.

* 질문: "Daniel은 빨간불에 신호를 무시하고 지나가도 되는 상황이었나요?"
* 질의 방식: 일반적인 질의

In [ ]:
# Create and send our query.

query = "Was Daniel allowed to pass at the red light?"

# Quick hack to reuse the same template with a different type of query.
prompt_template = template.format(claim=claim)
prompt = PromptTemplate(input_variables=["context", "question"], template=prompt_template)
conversation = prompt | llm

resp = conversation.invoke(input={"context": "", "question": query})

모델의 응답이 타당함을 확인할 수 있습니다. 이 경우 모델은 일반적인 교통 규제에 대한 지식을 기반으로 응답하고 있습니다.

### 두 번째 테스트: 추가 지식과 함께

이번에는 동일한 프롬프트와 질의를 사용하되,  
모델이 California Driver's Handbook에서 가져온 참조 정보를 함께 활용할 수 있도록 하겠습니다.

* 질문: "Daniel은 빨간불에 신호를 무시하고 지나가도 되는 상황이었나요?"
* 질의 방식: RetrievalQA

In [ ]:
# Create and send our query.

query = "Was Daniel allowed to pass at the red light?"

prompt_template = template.format(claim=claim)
prompt = PromptTemplate(input_variables=["context", "question"], template=prompt_template)
rag_chain = RetrievalQA.from_chain_type(
            llm,
            retriever=retriever,
            chain_type_kwargs={"prompt": prompt},
            return_source_documents=True,
        )
resp = rag_chain.invoke({"query": query})

정말 인상적입니다! 이제 모델이 반드시 지켜야 할 규칙들을 보다 구체적으로 언급하고 있습니다.

그런데 이 정보는 어디에서 온 것일까요?  
벡터 데이터베이스에서 가져온 답변에는 관련된 **출처(source)** 정보가 함께 포함되어 있으므로, 이를 확인해볼 수 있습니다.

In [ ]:
def format_sources(input_list):
    sources = ""
    if len(input_list) != 0:
        sources += input_list[0].metadata["metadata"]["source"] + ', page: ' + str(input_list[0].metadata["metadata"]["page"])
        page_list = [input_list[0].metadata["metadata"]["page"]]
        for item in input_list:
            if item.metadata["metadata"]["page"] not in page_list: # Avoid duplicates
                page_list.append(item.metadata["metadata"]["page"])
                sources += ', ' + str(item.metadata["metadata"]["page"])
    return sources


results = format_sources(resp['source_documents'])

print(results)

이제 끝입니다!  
이제 우리는 LLM에 외부 지식을 보완하여 활용하는 방법을 익혔습니다!